# MESSAGEix-Pakistan — Baseline Scenario

This notebook provides an **interactive** interface for running and exploring the Pakistan baseline scenario. For automated/reproducible runs, use `scripts/run_baseline.py` instead.

**Run from the repository root** so that relative paths resolve correctly.

In [ ]:
from pathlib import Path

import ixmp
import message_ix
import matplotlib.pyplot as plt

from message_pak.postprocess import (
    unit_correction,
    plotter,
    plot,
    get_report_df,
    primary_energy_by_fuel_plot,
    demand_by_sector_plot,
    emission_plots,
    operation_investment_cost_plot,
)

%matplotlib inline

## Configuration

In [ ]:
# Paths — resolved relative to the repo root
ROOT = Path('.')  # run notebook from repo root
DATA_FILE = ROOT / 'data' / 'data_MESSAGE_PK.xlsx'
OUTPUT_DIR = ROOT / 'output'
(OUTPUT_DIR / 'plots').mkdir(parents=True, exist_ok=True)

# Scenario identifiers
MODEL_NAME = 'Pakistan Model'
SCENARIO_NAME = 'baseline_xlsx'

## 1. Load Platform and Create Scenario

In [ ]:
mp = ixmp.Platform()
scenario = message_ix.Scenario(mp, model=MODEL_NAME, scenario=SCENARIO_NAME, version='new')

## 2. Load Data from Excel

In [ ]:
scenario.read_excel(str(DATA_FILE), add_units=True, commit_steps=False, init_items=True)

## 3. Correct Units

In [ ]:
unit_correction(mp, scenario)

## 4. Commit Scenario

In [ ]:
scenario.check_out()
scenario.commit(comment='Baseline scenario built from Excel data')
scenario.set_as_default()
print(f'Scenario version: {scenario.version}')

## 5. Solve

In [ ]:
case_name = f'{scenario.model}__{scenario.scenario}__v{scenario.version}'
scenario.solve()
print(f'Objective value: {scenario.var("OBJ")["lvl"]}')

## 6. Postprocessing — Stacked Bar Charts

In [ ]:
alldf = plotter(scenario, case_name, OUTPUT_DIR)
plot(alldf, case_name, OUTPUT_DIR)

## 7. Reporter — IAMC Format Plots

In [ ]:
pyam_df = get_report_df(scenario, output_csv=OUTPUT_DIR / 'MESSAGE_Pakistan_report.csv')

In [ ]:
primary_energy_by_fuel_plot(pyam_df)

In [ ]:
demand_by_sector_plot(pyam_df)

In [ ]:
emission_plots(pyam_df)

In [ ]:
operation_investment_cost_plot(pyam_df)

## 8. Close Database

In [ ]:
mp.close_db()